# 🧪 Model Evaluation via the vLLM / NIM Endpoint
This notebook performs an automated evaluation of your trained model. Test data is read line by line, sent to the local API endpoint, and the predictions are compared against the ground-truth labels.

In [ ]:
import json
import re
import requests

# ==========================================
# 1. Configuration & Paths
# ==========================================
# Path to the validation dataset (JSONL format)
VAL_FILE = "/data/nemo-fraud-detection/data/sft/validation_new.jsonl"

# Local API endpoint (vLLM / NIM Completions interface)
NIM_URL = "http://localhost:8000/v1/completions"

## ⚙️ Core Evaluation Function
The `run_evaluation()` function handles the following tasks:
1. **Read data:** Opens the JSONL validation dataset line by line.
2. **Prompt formatting:** Applies the exact training template (`### Instruction:` / `### Response:`).
3. **API request:** Sends the payload to the local server.
4. **Robust parsing:** Filters the model response using regular expressions (regex) for the target classes (`fraud` or `legitimate`).
5. **Metric calculation:** Counts correct predictions and prints a final accuracy summary (`Accuracy`).

In [ ]:
def run_evaluation():
    print(f"🚀 Starting evaluation via the vLLM Completions endpoint...\n")
    
    correct = 0
    total = 0
    
    with open(VAL_FILE, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            
            input_content = data.get("input", "")
            true_label = data.get("output", "").strip().lower()
            
            if not input_content:
                continue
                
            # Exact training template as a single string
            prompt_text = f"### Instruction:\n{input_content}\n\n### Response:\n"
            
            payload = {
                "model": "fraud-detection",
                "prompt": prompt_text,
                "temperature": 0.1,
                "max_tokens": 10
            }
            
            try:
                response = requests.post(NIM_URL, json=payload)
                result = response.json()
                
                if "choices" not in result:
                    print(f"⚠️ API error on entry {total + 1}: {result}")
                    continue

                # For /v1/completions, the generated text is located directly under "text"
                model_raw = result["choices"][0]["text"].strip().lower()
                
                # Robust parsing via regex
                if "legitimate" in model_raw and "fraud" not in model_raw:
                    model_answer = "legitimate"
                elif "fraud" in model_raw and "legitimate" not in model_raw:
                    model_answer = "fraud"
                else:
                    match = re.search(r'\b(fraud|legitimate)\b', model_raw)
                    model_answer = match.group(1) if match else model_raw
                
                is_correct = (true_label == model_answer)
                if is_correct:
                    correct += 1
                total += 1
                
                print(f"Entry {total} | Expected: {true_label} | Predicted: {model_answer} | Correct: {is_correct}")
                
            except Exception as e:
                print(f"❌ Error on entry {total + 1}: {e}")

    accuracy = (correct / total) * 100 if total > 0 else 0
    print("\n" + "="*40)
    print("📊 FINAL EVALUATION RESULTS")
    print("="*40)
    print(f"✅ Correctly predicted: {correct} / {total}")
    print(f"🎯 Accuracy:            {accuracy:.2f}%")
    print("="*40)


## ▶️ Execution
Run the cell below to start the evaluation run. Make sure the vLLM/NIM server is running in the background on port `8000`.

In [ ]:
if __name__ == "__main__":
    run_evaluation()